In [ ]:
import json
import os
import subprocess
import itertools
from datetime import datetime
import pandas as pd
from pathlib import Path
import sys
import pickle as pl
import torch
import itertools
import random

In [ ]:
env = os.environ.copy()
env['MKL_SERVICE_FORCE_INTEL'] = '1'
env['MKL_THREADING_LAYER'] = 'GNU'
env['OMP_NUM_THREADS'] = '1'

In [ ]:
with open('results_datainfo/snli_(0.1, 0.73, 0.17)/accuracy_arr.pkl','rb') as f:
   obj = pl.load(f)

In [ ]:
import random
import matplotlib.pyplot as plt

def sample_bimodal_beta(n=5000, p=0.5, a=2, b=90):
    """
    Mixture: with prob p sample Beta(a,b) (peaks near 0),
             else sample Beta(b,a) (peaks near 1).
    """
    xs = []
    for _ in range(n):
        if random.random() < p:
            xs.append(random.betavariate(a, b))
        else:
            xs.append(random.betavariate(b, a))
    return xs

# 1) sample
x = sample_bimodal_beta(n=6000, p=0.5, a=2, b=14)

# 2) "random points plot" (strip plot)
#    put points at y=0 with a tiny vertical jitter so you can see density
y = [(random.random() - 0.5) * 0.08 for _ in x]  # jitter in [-0.04, 0.04]

plt.figure()
plt.scatter(x, y, s=3, alpha=0.25)
plt.yticks([])
plt.xlim(0, 1)
plt.title("Random point plot (bimodal mixture on [0,1])")
plt.xlabel("x")
plt.show()

# 3) histogram view (often clearer)
plt.figure()
plt.hist(x, bins=80, density=True)
plt.xlim(0, 1)
plt.title("Histogram (density)")
plt.xlabel("x")
plt.ylabel("density")
plt.show()

In [ ]:
# computed_vectors = [[0.3, 0.25, 0.45],
# [0.3, 0.3, 0.4],
# [0.3, 0.35, 0.35] ,
# [0.3, 0.4, 0.3]  ,
# [0.3, 0.45, 0.25] ,
# [0.25, 0.3, 0.45]  ,
# [0.35, 0.3, 0.35], 
# [0.4, 0.3, 0.3] ,
# [0.45, 0.3, 0.25] ,
# [0.25, 0.45, 0.3]  ,
# [0.35, 0.35, 0.3]  ,
# [0.45, 0.25, 0.3]]

In [ ]:
import re 
def parse_proportion_from_filename(filename):
    """Extract proportion array from filename like 'ag_news_6e-06_[0.1, 0.3, 0.3, 0.3].json'"""
    match = re.search(r'[\[\(]\s*([0-9eE+\-.,\s]+)\s*[\]\)]', filename)
    if match:
        arr = [float(x) for x in match.group(1).split(",") if x.strip()]
        return arr
    return None


def load_experiment_data(dataset_name,interpolation_name,base_dir):
    """
    Load all experiment results with their proportions.
    
    Args:
        base_dir: Path to directory containing experiment folders
        
    Returns:
        list: List of dicts with keys 'proportions', 'alignment_matrix', 'labels', 'output_dir'
    """
    base_path = Path(base_dir)
    
    # Find all directories that match the pattern
    all_dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith(dataset_name) and interpolation_name in d.name]
    
    print(f"Scanning {len(all_dirs)} directories...")
    
    proportion_arr = []
    for output_dir in all_dirs:
        # Try to parse proportions from directory name
        proportion = parse_proportion_from_filename(output_dir.name)
        proportion_arr.append(proportion)
    return proportion_arr

In [ ]:
import math

In [ ]:
def generate_vectors(dataset_name,interpolation_name,base_dir,num_classes,no_of_examples):
    
    proportion_arr = load_experiment_data(dataset_name,interpolation_name,base_dir)
    vectors = []
    seen = set()  # Track unique combinations using sorted tuples
    num_classes = 3
    indices = list(range(num_classes))
    
    for i in range(no_of_examples):
        
        v = [ round(sample_bimodal_beta(n=1, p=0.5, a=2, b=10)[0],2) for _ in range(num_classes)]
        # v = [ round(random.uniform(0,1),2) for _ in range(num_classes)]
        sum_v = float(sum(v))
        v = [   round(i/sum_v ,2) for i in v]
    
        # Check if this combination (in sorted form) already exists
        v = tuple(v)
        if v not in seen and list(v) not in proportion_arr:
            seen.add(v)
            vectors.append(v)

    # print(f"Total unique combinations: {len(vectors)}")
    # for i, v in enumerate(vectors, 1):
    #     print(f"{i:2d}. {v}  (sum = {sum(v):.2f})")
        
    return vectors

In [ ]:
# def generate_vectors(dataset_name,interpolation_name,base_dir,step,fixed_value,max_value,min_value,num_classes):
    
#     proportion_arr = load_experiment_data(dataset_name,interpolation_name,base_dir)
#     remaining_sum = 1.0 - fixed_value
    
#     values = [round(step * i, 2) for i in range( math.ceil(min_value/step),   int(max_value/step) + 1)]
#     print(values)
#     vectors = []
#     seen = set()  # Track unique combinations using sorted tuples
#     num_classes = 3
#     indices = list(range(num_classes))

#     for fixed_idx in indices:  # Which position to fix at 0.3
#         for x in values:
#             y = round(remaining_sum - x, 2)
            
#             # Only include if y is valid (0.05 <= y <= 0.65)
#             if min_value <= y <= max_value:
#                 v = [0.0] * num_classes
#                 v[fixed_idx] = fixed_value
                
#                 # Fill other two positions
#                 other_indices = [idx for idx in indices if idx != fixed_idx]
#                 v[other_indices[0]] = x
#                 v[other_indices[1]] = y
                
#                 # Check if this combination (in sorted form) already exists
#                 v = tuple(v)
#                 if v not in seen and list(v) not in proportion_arr:
#                     seen.add(v)
#                     vectors.append(v)

#     # print(f"Total unique combinations: {len(vectors)}")
#     # for i, v in enumerate(vectors, 1):
#     #     print(f"{i:2d}. {v}  (sum = {sum(v):.2f})")
        
#     return vectors

In [ ]:
step = 0.07
fixed_value = 0.3  # One class fixed at this 
max_value = 0.65
min_value = 0.05
fixed_values_arr = [0.3,0.35,0.4]

In [ ]:
# vectors_set = set()
# for fixed_value in fixed_values_arr:
#     vectors_set.update(generate_vectors('snli','linear','./results1',step,fixed_value,max_value,min_value,num_classes=3))

In [ ]:
vectors_set = set()
for interpolation in ['linear','model_baseline','slerp','ties']:
    vectors_set.update(generate_vectors('snli',interpolation,'/home/aditya/hack_model/results_align_matrix',3,50))

In [ ]:
vectors_list = list(vectors_set)

In [ ]:
vectors_list

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def distance_of_proportions(X, ddof=0):
    """
    X: array shape (N,3) with values in [0,1]
    returns:
      P: normalized proportions, shape (N,3)
    """
    X = np.asarray(X, dtype=float)
    P = X / X.sum(axis=1, keepdims=True)
    l1 = np.sum(np.abs(P - 1/3), axis=1)
    return l1

# Example usage: suppose X is your generated data, shape (N,3)
# X = np.random.rand(10000, 3)

l1 = distance_of_proportions(vectors_list, ddof=0)

plt.figure()
plt.hist(l1, bins=80, density=True)  # density=True => area under histogram = 1
plt.xlabel("sum of difference of proportion")
plt.ylabel("density")
plt.title("Distribution of distance of normalized proportions from balanced proportion")
plt.show()

In [ ]:
# 0.000006 - stable learning rate

parameter_grid_2 = {
    "n_total": [5000],
    "n_finetune": [2500],
    "model_name": ["bert-base-uncased"],
    "max_length": [256],
    "num_labels": [3],
    "batch_size": [32],
    "learning_rate": [0.000006],
    "num_epochs": [7],
    "K": [15],
    "lambda_min": [0.05],
    "lambda_max": [0.95],
    "interpolations": [["linear","model_baseline","slerp","ties"],],
    "optimizer": ["Adam"],
    "dataset": ["snli"],
    "proportionArr": vectors_list,
    "finetuning_source": ["original"]
}

# 0.000006, 0.00001

# # to be fixed
# [0.1,0.7,0.1,0.1] 

# # done 
# [0.25,0.25,0.25,0.25]
# [0.3,0.1,0.3,0.3]


    # "learning_rate": [0.000006,0.000003],



In [ ]:
# parameter_grid_3 = {
#     "n_total": [5000],
#     "n_finetune": [2500],
#     "model_name": ["bert-base-uncased"],
#     "max_length": [256],
#     "num_labels": [5],
#     "batch_size": [32],
#     "learning_rate": [0.001,0.003],
#     "num_epochs": [2],
#     "K": [15],
#     "lambda_min": [0.05],
#     "lambda_max": [0.95],
#     "interpolation": ["linear"],
#     "optimizer": ["Adam"],
#     "dataset": ["yelp_review_full"],
#     "proportionArr": [[0.27,0.1,0.27,0.1,0.26],]
    
    
    # # to be fixed
    # [0.1,0.1,0.6,0.1,0.1]
    # [0.05,0.1,0.05,0.7,0.1]
    
    
    # # done 
    # [0.2,0.2,0.2,0.2,0.2]


In [ ]:
parameter_grids = []
parameter_grids.append(parameter_grid_2)
# parameter_grids.append(parameter_grid_3)

In [ ]:
# Or define specific combinations
specific_configs = []

In [ ]:
def generate_all_combinations(param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = []
    
    for combination in itertools.product(*values):
        config = dict(zip(keys, combination))
        combinations.append(config)
    
    return combinations

def create_config_file(config, experiment_path):
    config['experiment_name'] = experiment_path
    config_path = experiment_path + ".json"
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    print(f"Created config file: {config_path}")

def run_experiment(config, experiment_name):
    try:
        # Create config file
        create_config_file(config, f"{experiment_name}")
        
        print(f"\nRunning experiment: {experiment_name}")
        print(f"Config: {config}")
        
        # result = subprocess.run([sys.executable, "agnews_sample_hacking_last_layer.py", f"{experiment_name}.json"], capture_output=True, text=True,shell=True)
        result = subprocess.call([sys.executable, "domain_distribution.py", f"{experiment_name}.json"],env=env)
        
        if result == 0:
            print(f"✅ Experiment {experiment_name} completed successfully")
        else:
            print(f"❌ Experiment {experiment_name} failed")
            # print("Error:", result.stderr)
        
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": result == 0,
            # "stdout": result.stdout,
            # "stderr": result.stderr
        }
        
    except Exception as e:
        print(f"❌ Exception in {experiment_name}: {str(e)}")
        return {
            "experiment_name": experiment_name,
            "config": config,
            "success": False,
            "error": str(e)
        }

In [ ]:
configs_to_run = generate_all_combinations(parameter_grids[0])
len(configs_to_run)

In [ ]:
configs_to_run[0]

In [ ]:
# Choose experiment mode
USE_GRID_SEARCH = True # Set to True for grid search, False for specific configs

if USE_GRID_SEARCH:
    for parameter_grid in parameter_grids:
        configs_to_run = generate_all_combinations(parameter_grid)
        print(f"Total configurations to run: {len(configs_to_run)}")
        
        # Run all experiments
        results = []
        for i, config in enumerate(configs_to_run):
            experiment_name = f"{config['dataset']}_{config['proportionArr']}"
            result = run_experiment(config, experiment_name)
            results.append(result)

        # Summary
        successful = sum(1 for r in results if r["success"])
        print(f"\n{'='*50}")
        print(f"EXPERIMENT SUMMARY")
        print(f"{'='*50}")
        print(f"Total experiments: {len(results)}")
        print(f"Successful: {successful}")
        print(f"Failed: {len(results) - successful}")
        
else:
    configs_to_run = specific_configs

In [ ]:
# When a parent process starts a child process via subprocess, the two are separate entities with their own memory space.
# The parent process is not notified in real-time about the filesystem modifications the child process is making.